# Chat ou Chien !! une méthode d'apprentissage automatique.

Dans une entreprise de marketing, on veut faire une étude sur les clients qui fréquentent un grand magasin. L'idée est d'estimer le pourcentage de clients qui ont des chats et ceux qui ont des chiens, afin de prendre des décisions marketing ciblées. Vous proposez d'utiliser une caméra pour détecter et compter les deux animaux dans le magasin. La première étape du projet est de développer un modèle qui peut détecter le chat ou le chien. Ainsi, dans ce brief, en utilisant une base de données, vous allez entrainer un modèle d'apprentissage automatique avec un apprentissage supervisé.

# Veille technologique: Opencv python

- Préparation données imagerie pour méthode classique d'apprentissage automatique
- Entrainement et évaluation (similaire aux briefs précédents)

## Operations Simples

In [21]:
# use opencv to load and display the image
import os
import cv2
import numpy as np

In [22]:
# Préparer data 
size = 64  # à la base c'était 150, mais ca prenait toute la RAM
# repertoir d'images avec deux sous dossiers "Cat" et "Dog"
image_directory = 'C:\\Users\\julyc\\OneDrive\\Bureau\\Stage\\POC\\PetImages\\'
images = []  # liste pour images  
label = []  # liste pour Label (0 ou 1) pour deux classes.

# utiliser "os" pour avoir les noms des images dans chaque sous dossier
cat_images = [f for f in os.listdir(image_directory + 'Cat\\') if f.endswith('.jpg')]
dog_images = [f for f in os.listdir(image_directory + 'Dog\\') if f.endswith('.jpg')]

print(f"Cats trouvés : {len(cat_images)}")
print(f"Dogs trouvés : {len(dog_images)}")

# utiliser une boucle pour lire chaque image, redimensionner en (150,150,3),
# et la mettre dans images, et mettre le label(0 ou 1) selon le type dans "label"

# il y a des images corrumpues, utiliser try ... catch except pour les ignorer


# boucle pour les images Cat label = 0
for img_name in cat_images:
    try:
        img = cv2.imread(image_directory + 'Cat\\' + img_name)
        img = cv2.resize(img, (size, size))
        images.append(img)
        label.append(0)
    except Exception as e:
        print(f"Image corrompue ignorée : {img_name}")

# boucle pour les images Dog label = 1
for img_name in dog_images:
    try:
        img  = cv2.imread(image_directory + 'Dog\\' + img_name)
        img = cv2.resize(img, (size, size))
        images.append(img)
        label.append(1)
    except Exception as e:
        print(f"Image corrompue ignorée : {img_name}")

# transformer les listes en numpy 
images = np.array(images)
label = np.array(label)

# save data as ".npy" file

np.save('image.npy', images)
np.save('label.npy', label)


# show shape

print(f"Images shape : {images.shape}")
print(f"Label shape : {label.shape}")

Cats trouvés : 12500
Dogs trouvés : 12499
Image corrompue ignorée : 10404.jpg
Image corrompue ignorée : 666.jpg
Image corrompue ignorée : 11702.jpg
Images shape : (24996, 64, 64, 3)
Label shape : (24996,)


In [23]:
# prepare data for classical machine learning model

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# utiliser reshape pour redimensionner les données images (exemple: (120,150,150,3) ---> (120,150*150*3)
X = images.reshape(len(images), size * size * 3)

# split les données en train et test avec "stratification"
X_train, X_test, y_train, y_test = train_test_split(
    X, label, test_size=0.2, random_state=42, stratify=label
)

In [24]:
# initialiser le classifier
clf = RandomForestClassifier(n_estimators=100, random_state=42)

# entrainer le classifier
clf.fit(X_train, y_train)

MemoryError: Unable to allocate 937. MiB for an array with shape (19996, 12288) and data type float32

In [ ]:
# evaluate classifier
from sklearn.metrics import classification_report
# predire
y_pred = clf.predict(X_test)

# evaluer avec classification_report
print(classification_report(y_test, y_pred, target_names=['Cat', 'Dog']))

              precision    recall  f1-score   support

         Cat       0.64      0.71      0.67      2500
         Dog       0.68      0.60      0.64      2500

    accuracy                           0.66      5000
   macro avg       0.66      0.66      0.65      5000
weighted avg       0.66      0.66      0.65      5000



### Essayer de normaliser chaque image entre 0 et 255 et rentrainer le modèle. Y a-t-il une amélioration ?

In [ ]:
import numpy as np
import cv2

X = np.load('image.npy')
label = np.load('label.npy')
size = 64

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

images_norm = []
for img in images:
    img_norm = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX)
    images_norm.append(img_norm)

images_norm = np.array(images_norm)

# Reshape
X_norm = images_norm.reshape(len(images_norm), size * size * 3)

# Split
X_train_n, X_test_n, y_train_n, y_test_n = train_test_split(
    X_norm, label, test_size=0.2, random_state=42, stratify=label
)

# Entraîner
clf_norm = RandomForestClassifier(n_estimators=100, random_state=42)
clf_norm.fit(X_train_n, y_train_n)

# Évaluer
y_pred_n = clf_norm.predict(X_test_n)
print("Avec normalisation :")
print(classification_report(y_test_n, y_pred_n, target_names=['Cat', 'Dog']))

Avec normalisation :
              precision    recall  f1-score   support

         Cat       0.64      0.71      0.67      2500
         Dog       0.67      0.61      0.64      2500

    accuracy                           0.66      5000
   macro avg       0.66      0.66      0.66      5000
weighted avg       0.66      0.66      0.66      5000



Correction Bassam pour la normalisation

In [ ]:
import cv2
import numpy as np
from matplotlib import pyplot as plt
 
img = cv2.imread('100.jpg',0)
 
hist,bins = np.histogram(img.flatten(),256,[0,256])
 
cdf = hist.cumsum()
cdf_normalized = cdf * hist.max()/ cdf.max()
 
plt.plot(cdf_normalized, color = 'b')
plt.hist(img.flatten(),256,[0,256], color = 'r')
plt.xlim([0,256])
plt.legend(('cdf','histogram'), loc = 'upper left')
plt.show()

# Neural networ MLP

Entrainer un modèle MLP pour detecter la class Dog ou Cat.

In [ ]:
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
import numpy as np

# Charger les donnees depuis les fichiers sauvegardes
images = np.load('image.npy')
label = np.load('label.npy')
size = 64

# Reshape et normalisation entre 0 et 1
X = images.reshape(len(images), size * size * 3).astype(np.float32) / 255.0

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, label, test_size=0.2, random_state=42, stratify=label
)

# Creer le MLP
inputs = Input(shape=(size * size * 3,))
x = Dense(128, activation='relu')(inputs)
x = Dense(128, activation='relu')(x)
outputs = Dense(1, activation='sigmoid')(x)

model = Model(inputs=inputs, outputs=outputs)

model.compile(
    optimizer=Adam(),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

In [ ]:
# entrainer le modèle avec validation_split = 0.2, epochs = 20, et un batch_size = 128
history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=128
)

In [ ]:
# evaluate MLP
# evaluate classifier
from sklearn.metrics import classification_report
# predire
y_pred_proba = model.predict(X_test)
y_pred = (y_pred_proba > 0.5).astype(int).flatten()

# evaluer avec classification_report
print(classification_report(y_test, y_pred, target_names=['Cat', 'Dog']))